In [1]:
# ============================================================
# MTN NIGERIA PREPAID CHURN PREDICTION
# Phase 1 — Data Generation & Exploratory Data Analysis
# ============================================================
# Author: Elijjjaaaahhhhh
# Dataset: NGA_MTN_Subscribers_500K (synthetic)
# Seed: 42 (all random operations use this seed for reproducibility)
# ============================================================

# --- Core data libraries ---
import numpy as np
import pandas as pd

# --- Visualisation libraries ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Display settings ---
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

# --- Reproducibility ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("All libraries imported successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("Random seed set to 42 — all results are reproducible.")

All libraries imported successfully.
NumPy version: 2.4.3
Pandas version: 3.0.1
Random seed set to 42 — all results are reproducible.


In [2]:
# ============================================================
# CELL 2 — DATASET PARAMETERS
# Define all constants that control dataset generation.
# Change values here — nowhere else — to adjust the dataset.
# ============================================================

# --- Scale ---
N_SUBSCRIBERS = 500_000       # Total number of rows to generate
N_MONTHS      = 36            # Jan 2022 – Dec 2024

# --- Churn rate ---
BASE_MONTHLY_CHURN_RATE = 0.05   # 5% monthly churn (~25,000 churners)

# --- Nigerian states (36 states + FCT = 37 total) ---
NIGERIAN_STATES = [
    'Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa',
    'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo',
    'Ekiti', 'Enugu', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano',
    'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger',
    'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto',
    'Taraba', 'Yobe', 'Zamfara', 'FCT Abuja'
]

# --- States with higher churn (more competition, higher income) ---
HIGH_CHURN_STATES = ['Lagos', 'Kano', 'FCT Abuja', 'Rivers', 'Enugu']

# --- MTN Nigeria tariff plans ---
TARIFF_PLANS = ['MTNPulse', 'BetaGist', 'XtraValue', 'XtraSpecial', 'mPulse']

# --- Tariff plan weights (how common each plan is in the subscriber base) ---
# TalkMore is the most common prepaid plan
# SMEBundle is least common — it targets small businesses
TARIFF_WEIGHTS = [0.35, 0.15, 0.25, 0.15, 0.10]  # Must sum to 1.0

# --- State distribution weights ---
# Lagos, Kano, and Abuja have the largest subscriber bases
# We build this dynamically so every state gets a weight
STATE_WEIGHTS = []
for state in NIGERIAN_STATES:
    if state == 'Lagos':
        STATE_WEIGHTS.append(0.18)      # Lagos alone = 18% of subscribers
    elif state == 'Kano':
        STATE_WEIGHTS.append(0.07)      # Kano = 7%
    elif state == 'FCT Abuja':
        STATE_WEIGHTS.append(0.10)      # Abuja = 10%
    elif state == 'Rivers':
        STATE_WEIGHTS.append(0.06)      # Rivers = 6%
    elif state == 'Enugu':
        STATE_WEIGHTS.append(0.06)      # Enugu = 6%
    else:
        # Remaining 32 states share the remaining 53%
        STATE_WEIGHTS.append(0.53 / 32)

# Confirm weights sum to 1.0 (they must — this is a probability distribution)
assert round(sum(STATE_WEIGHTS), 6) == 1.0, "State weights must sum to 1.0"

# --- Observation months ---
import pandas as pd
OBSERVATION_MONTHS = pd.date_range(start='2022-01-01', periods=N_MONTHS, freq='MS')
# 'MS' = Month Start — generates the first day of each month

# --- Print confirmation ---
print(f"Dataset parameters defined.")
print(f"Total subscribers     : {N_SUBSCRIBERS:,}")
print(f"Observation period    : {OBSERVATION_MONTHS[0].strftime('%b %Y')} to {OBSERVATION_MONTHS[-1].strftime('%b %Y')}")
print(f"Number of months      : {N_MONTHS}")
print(f"Base monthly churn    : {BASE_MONTHLY_CHURN_RATE*100:.0f}%")
print(f"Number of states      : {len(NIGERIAN_STATES)}")
print(f"Tariff plans          : {TARIFF_PLANS}")
print(f"State weights sum     : {sum(STATE_WEIGHTS):.6f} ✓")


Dataset parameters defined.
Total subscribers     : 500,000
Observation period    : Jan 2022 to Dec 2024
Number of months      : 36
Base monthly churn    : 5%
Number of states      : 37
Tariff plans          : ['MTNPulse', 'BetaGist', 'XtraValue', 'XtraSpecial', 'mPulse']
State weights sum     : 1.000000 ✓


In [4]:
# ============================================================
# CELL 3 — DATASET GENERATION (FINAL VERSION)
# Generates 500,000 synthetic MTN Nigeria subscriber rows.
# Schema: 21 columns
# Churn labels derived from behaviour — not randomly assigned.
#
# USSD codes:
#   *310# = airtime balance  | *323# = data balance
#   *312# = buy data bundle  | *303# = borrow airtime
#
# Subscriber segments:
#   Traditional (45%) — voice-first, USSD, scratch card
#   Digital     (55%) — data-first, app, bank transfer
#
# Conditional signal logic:
#   Signals are context-aware — severity depends on combination
#   of features, not flat penalties. See each signal for full
#   documented logic.
# ============================================================

def generate_mtn_dataset(n_subscribers, random_seed=42):
    """
    Generates a synthetic MTN Nigeria prepaid subscriber dataset.

    INPUT:
        n_subscribers (int): Number of rows to generate
        random_seed   (int): Seed for reproducibility — default 42

    OUTPUT:
        pandas DataFrame with 21 columns and n_subscribers rows

    CHURN LOGIC:
        Churn probability is calculated from 19 behavioural signals.
        Signals are conditional — severity depends on feature combinations.
        Final binary label is drawn from that probability.
        Order of operations: additive signals → multipliers → 
        segment signals → seasonal → state → clip → draw label.
    """

    np.random.seed(random_seed)
    print(f"Generating {n_subscribers:,} subscriber records...")
    print("This may take 30–60 seconds. Please wait.")

    # --------------------------------------------------------
    # SECTION 1 — SUBSCRIBER IDs AND OBSERVATION MONTHS
    # --------------------------------------------------------

    subscriber_ids = [
        f"MTN-{np.random.randint(10000000, 99999999)}"
        for _ in range(n_subscribers)
    ]

    observation_months = np.random.choice(
        OBSERVATION_MONTHS, size=n_subscribers
    )

    # --------------------------------------------------------
    # SECTION 2 — SUBSCRIBER SEGMENT ASSIGNMENT
    # 45% Traditional | 55% Digital
    # Drives realistic behaviour differences across 7 columns
    # --------------------------------------------------------

    subscriber_segment = np.random.choice(
        ['Traditional', 'Digital'],
        size=n_subscribers,
        p=[0.45, 0.55]
    )

    # Boolean masks — used throughout to apply segment logic cleanly
    is_traditional = (subscriber_segment == 'Traditional')
    is_digital     = (subscriber_segment == 'Digital')

    # --------------------------------------------------------
    # SECTION 3 — STATE AND TARIFF PLAN
    # Assigned early — needed for state-aware channel logic
    # --------------------------------------------------------

    states = np.random.choice(
        NIGERIAN_STATES,
        size=n_subscribers,
        p=STATE_WEIGHTS
    )

    tariff_plans = np.random.choice(
        TARIFF_PLANS,
        size=n_subscribers,
        p=TARIFF_WEIGHTS
    )

    # Boolean mask for high-churn urban states
    # Used to reduce scratch card prevalence in urban areas
    # and to adjust state-level churn signals
    is_urban_state = np.isin(states, HIGH_CHURN_STATES)

    # --------------------------------------------------------
    # SECTION 4 — DAYS SINCE LAST RECHARGE
    # Exponential distribution — most subscribers recharged recently
    # Traditional: scale=18 (slightly more dormant on average)
    # Digital:     scale=12 (more active, habitual recharging)
    # --------------------------------------------------------

    days_since_last_recharge = np.where(
        is_traditional,
        np.clip(
            np.random.exponential(scale=18, size=n_subscribers).astype(int),
            0, 90
        ),
        np.clip(
            np.random.exponential(scale=12, size=n_subscribers).astype(int),
            0, 90
        )
    )

    # --------------------------------------------------------
    # SECTION 5 — RECHARGE FREQUENCY
    # Poisson — count of recharge events in 30-day window
    # Traditional: recharges more often (lam=6) in smaller amounts
    # Digital:     recharges less often (lam=4) in larger amounts
    # --------------------------------------------------------

    recharge_frequency_30d = np.where(
        is_traditional,
        np.clip(np.random.poisson(lam=6, size=n_subscribers), 0, 30),
        np.clip(np.random.poisson(lam=4, size=n_subscribers), 0, 30)
    )

    # --------------------------------------------------------
    # SECTION 6 — AVERAGE RECHARGE VALUE
    # Traditional: small frequent top-ups ₦100–₦400
    #   Mass market (90%): normal loc=250, scale=100
    #   High value  (10%): normal loc=800, scale=200
    # Digital: larger bundle purchases ₦500–₦3,000
    #   Mass market (80%): normal loc=800, scale=300
    #   High value  (20%): normal loc=2500, scale=800
    # --------------------------------------------------------

    trad_mass     = np.random.normal(loc=250, scale=100, size=n_subscribers)
    trad_high     = np.random.normal(loc=800, scale=200, size=n_subscribers)
    trad_flag     = np.random.binomial(1, 0.10, size=n_subscribers)
    trad_recharge = np.where(trad_flag == 1, trad_high, trad_mass)

    dig_mass      = np.random.normal(loc=800,  scale=300, size=n_subscribers)
    dig_high      = np.random.normal(loc=2500, scale=800, size=n_subscribers)
    dig_flag      = np.random.binomial(1, 0.20, size=n_subscribers)
    dig_recharge  = np.where(dig_flag == 1, dig_high, dig_mass)

    avg_recharge_value_naira = np.where(
        is_traditional, trad_recharge, dig_recharge
    )
    avg_recharge_value_naira = np.clip(
        avg_recharge_value_naira, 50, 5000
    ).round(2)

    # --------------------------------------------------------
    # SECTION 7 — USSD SESSIONS
    # NCC-harmonised codes: *310# *323# *312# *303#
    # Traditional: primary self-service channel (lam=30)
    # Digital:     uses app instead (lam=8)
    # --------------------------------------------------------

    ussd_sessions_30d = np.where(
        is_traditional,
        np.clip(np.random.poisson(lam=30, size=n_subscribers), 0, 200),
        np.clip(np.random.poisson(lam=8,  size=n_subscribers), 0, 200)
    )

    # --------------------------------------------------------
    # SECTION 8 — APP SESSIONS (MyMTN App)
    # Traditional: rarely uses app (lam=0.5)
    # Digital:     regular app user (lam=12)
    # Weak churn signal — meaningful only when combined with
    # low data and low frequency (Signal 14D stacked logic)
    # --------------------------------------------------------

    app_sessions_30d = np.where(
        is_traditional,
        np.clip(np.random.poisson(lam=0.5, size=n_subscribers), 0, 5),
        np.clip(np.random.poisson(lam=12,  size=n_subscribers), 0, 50)
    )

    # --------------------------------------------------------
    # SECTION 9 — DATA CONSUMPTION
    # Traditional: low or zero — 0 to 2,000MB
    # Digital:     heavy user — 0 to 50,000MB (50GB ceiling)
    # --------------------------------------------------------

    data_mb_consumed_30d = np.where(
        is_traditional,
        np.clip(
            np.random.exponential(scale=300,  size=n_subscribers).round(2),
            0, 2000
        ),
        np.clip(
            np.random.exponential(scale=6000, size=n_subscribers).round(2),
            0, 50000
        )
    )

    # --------------------------------------------------------
    # SECTION 10 — VOICE MINUTES
    # Traditional: primary communication channel (loc=300)
    # Digital:     uses WhatsApp calls (loc=100)
    # --------------------------------------------------------

    voice_minutes_30d = np.where(
        is_traditional,
        np.clip(
            np.random.normal(loc=300, scale=120, size=n_subscribers).round(2),
            0, 1000
        ),
        np.clip(
            np.random.normal(loc=100, scale=80, size=n_subscribers).round(2),
            0, 1000
        )
    )

    # --------------------------------------------------------
    # SECTION 11 — SMS COUNT
    # Traditional: still uses SMS regularly (lam=25)
    # Digital:     WhatsApp dominant — SMS nearly irrelevant (lam=5)
    # --------------------------------------------------------

    sms_count_30d = np.where(
        is_traditional,
        np.clip(np.random.poisson(lam=25, size=n_subscribers), 0, 300),
        np.clip(np.random.poisson(lam=5,  size=n_subscribers), 0, 300)
    )

    # --------------------------------------------------------
    # SECTION 12 — BINARY FLAGS
    # --------------------------------------------------------

    # network_complaint_flag: 8% base rate
    network_complaint_flag = np.random.binomial(1, 0.08, size=n_subscribers)

    # competitor_sim_detected:
    # Traditional: 10% | Digital: 22%
    competitor_sim_detected = np.where(
        is_traditional,
        np.random.binomial(1, 0.10, size=n_subscribers),
        np.random.binomial(1, 0.22, size=n_subscribers)
    )

    # number_portability_flag: 1 = ported IN from another network
    # Traditional: 8% | Digital: 16%
    number_portability_flag = np.where(
        is_traditional,
        np.random.binomial(1, 0.08, size=n_subscribers),
        np.random.binomial(1, 0.16, size=n_subscribers)
    )

    # --------------------------------------------------------
    # SECTION 13 — TENURE
    # Traditional: longer tenure (loc=48) — established subscribers
    # Digital:     shorter tenure (loc=24) — younger, more recent
    # --------------------------------------------------------

    tenure_months = np.where(
        is_traditional,
        np.clip(
            np.random.normal(loc=48, scale=24, size=n_subscribers).astype(int),
            1, 120
        ),
        np.clip(
            np.random.normal(loc=24, scale=18, size=n_subscribers).astype(int),
            1, 120
        )
    )

    # --------------------------------------------------------
    # SECTION 14 — RECHARGE CHANNEL
    # Boolean indexing used — avoids np.where evaluating both
    # arrays before applying condition (dtype and logic safety)
    #
    # Three sub-groups:
    #   Traditional rural  — scratch card still present (20%)
    #   Traditional urban  — scratch card minimal (5%)
    #   Digital            — app and bank transfer dominant
    # --------------------------------------------------------

    recharge_channel = np.empty(n_subscribers, dtype=object)

    # Traditional non-urban — scratch card present
    mask_trad_rural = is_traditional & ~is_urban_state
    n_trad_rural    = mask_trad_rural.sum()
    if n_trad_rural > 0:
        recharge_channel[mask_trad_rural] = np.random.choice(
            ['Scratch Card', 'Agent', 'USSD', 'Bank Transfer', 'MTN App'],
            size=n_trad_rural,
            p=[0.20, 0.30, 0.30, 0.15, 0.05]
        )

    # Traditional urban — scratch card minimal
    mask_trad_urban = is_traditional & is_urban_state
    n_trad_urban    = mask_trad_urban.sum()
    if n_trad_urban > 0:
        recharge_channel[mask_trad_urban] = np.random.choice(
            ['Scratch Card', 'Agent', 'USSD', 'Bank Transfer', 'MTN App'],
            size=n_trad_urban,
            p=[0.05, 0.25, 0.35, 0.25, 0.10]
        )

    # Digital — app and bank transfer dominant
    mask_digital = is_digital
    n_digital    = mask_digital.sum()
    if n_digital > 0:
        recharge_channel[mask_digital] = np.random.choice(
            ['Scratch Card', 'Agent', 'USSD', 'Bank Transfer', 'MTN App'],
            size=n_digital,
            p=[0.05, 0.10, 0.15, 0.45, 0.25]
        )

    # --------------------------------------------------------
    # SECTION 15 — BUNDLE TYPE
    # Boolean indexing — same reason as Section 14
    # Traditional: Voice Only 45% | Data Only 20% |
    #              Combo 30%      | Social 5%
    # Digital:     Voice Only 5%  | Data Only 45% |
    #              Combo 25%      | Social 25%
    # --------------------------------------------------------

    bundle_type = np.empty(n_subscribers, dtype=object)

    n_trad = is_traditional.sum()
    if n_trad > 0:
        bundle_type[is_traditional] = np.random.choice(
            ['Voice Only', 'Data Only', 'Voice+Data Combo', 'Social Media Bundle'],
            size=n_trad,
            p=[0.45, 0.20, 0.30, 0.05]
        )

    n_dig = is_digital.sum()
    if n_dig > 0:
        bundle_type[is_digital] = np.random.choice(
            ['Voice Only', 'Data Only', 'Voice+Data Combo', 'Social Media Bundle'],
            size=n_dig,
            p=[0.05, 0.45, 0.25, 0.25]
        )

    # --------------------------------------------------------
    # SECTION 16 — CUSTOMER SERVICE CONTACTS
    # Base rate: lam=0.8 (most subscribers never contact)
    # Complaint subscribers: lam=3.5 (contacted before escalating)
    # --------------------------------------------------------

    customer_service_contacts_30d = np.where(
        network_complaint_flag == 1,
        np.clip(np.random.poisson(lam=3.5, size=n_subscribers), 0, 10),
        np.clip(np.random.poisson(lam=0.8, size=n_subscribers), 0, 10)
    )

    # --------------------------------------------------------
    # SECTION 17 — CHURN PROBABILITY CALCULATION
    #
    # ORDER OF OPERATIONS:
    #   1. Base rate
    #   2. Recency signal (additive)
    #   3. Signal 2:  Recharge frequency (conditional additive)
    #   4. Signal 3:  Network complaint (multiplier)
    #   5. Signal 4:  Competitor SIM (tenure-decayed multiplier)
    #   6. Signal 5:  New subscriber (engagement conditional)
    #   7. Signal 6:  Number portability (multiplier)
    #   8. Signal 7:  Customer service (escalation conditional)
    #   9. Signal 8:  High value (subtractive)
    #  10. Signal 9T: USSD decay (traditional)
    #  11. Signal 10T:Voice + complaint (traditional)
    #  12. Signal 11T:Scratch card frequency decay (rural traditional)
    #  13. Signal 12D:Social media bundle (tiered digital)
    #  14. Signal 13D:Low app sessions (digital)
    #  15. Signal 14D:Low data stacked (digital)
    #  16. Seasonal adjustments
    #  17. State adjustments
    #  18. Clip → Draw label
    # --------------------------------------------------------

    # Base rate — everyone starts at 5% monthly churn
    churn_prob = np.full(n_subscribers, BASE_MONTHLY_CHURN_RATE)

    # --------------------------------------------------
    # SIGNAL 1: Days since last recharge
    # Strongest single predictor — linear scale
    # 90 days inactive = +35pp churn probability
    # At 45 days: +17.5pp | At 21 days: +8.2pp
    # --------------------------------------------------
    churn_prob += (days_since_last_recharge / 90) * 0.35

    # --------------------------------------------------
    # SIGNAL 2: Low recharge frequency — conditional
    #
    # Traditional + frequency < 3 + low value (<₦500)
    #   → +0.15 (genuinely disengaging)
    # Traditional + frequency < 3 + higher value (≥₦500)
    #   → +0.07 (low frequency but consolidating spends)
    # Digital + frequency < 3 + low value (<₦500)
    #   → +0.09 (low frequency AND low value = unmet expectation)
    # Digital + frequency < 3 + higher value (≥₦500)
    #   → +0.02 (bundle buyer — 2 recharges of ₦1,500 is normal)
    # --------------------------------------------------
    low_freq = recharge_frequency_30d < 3
    low_val  = avg_recharge_value_naira < 500

    churn_prob += np.where(
        is_traditional & low_freq & low_val,  0.15, 0.0)
    churn_prob += np.where(
        is_traditional & low_freq & ~low_val, 0.07, 0.0)
    churn_prob += np.where(
        is_digital & low_freq & low_val,      0.09, 0.0)
    churn_prob += np.where(
        is_digital & low_freq & ~low_val,     0.02, 0.0)

    # --------------------------------------------------
    # SIGNAL 3: Network complaint — 2.3x multiplier
    # Subscriber frustrated enough to formally complain
    # Compounds all signals accumulated so far
    # --------------------------------------------------
    churn_prob *= np.where(network_complaint_flag == 1, 2.3, 1.0)

    # --------------------------------------------------
    # SIGNAL 4: Competitor SIM — tenure-decayed multiplier
    #
    # tenure < 12 months  → 2.2x (no loyalty, already dual-SIM)
    # tenure 12–36 months → 1.8x (moderate risk — current baseline)
    # tenure 37–60 months → 1.3x (established — SIM likely supplementary)
    # tenure > 60 months  → 1.1x (5+ years on MTN — not leaving)
    # no competitor SIM   → 1.0x (no change)
    # --------------------------------------------------
    has_competitor = competitor_sim_detected == 1

    competitor_multiplier = np.where(
        has_competitor & (tenure_months < 12),          2.2,
        np.where(
        has_competitor & (tenure_months < 37),          1.8,
        np.where(
        has_competitor & (tenure_months < 61),          1.3,
        np.where(
        has_competitor & (tenure_months >= 61),         1.1,
        1.0))))

    churn_prob *= competitor_multiplier

    # --------------------------------------------------
    # SIGNAL 5: New subscriber — engagement conditional
    #
    # tenure <= 6 AND frequency < 3 AND ussd < 5
    #   → +0.15 (signed up and immediately went quiet)
    # tenure <= 6 AND frequency >= 3 AND frequency < 6
    #   → +0.05 (new but showing some engagement)
    # tenure <= 6 AND frequency >= 6
    #   → +0.01 (new and highly engaged — settling in well)
    # --------------------------------------------------
    is_new = tenure_months <= 6

    churn_prob += np.where(
        is_new & (recharge_frequency_30d < 3) & (ussd_sessions_30d < 5),
        0.15, 0.0
    )
    churn_prob += np.where(
        is_new & (recharge_frequency_30d >= 3) & (recharge_frequency_30d < 6),
        0.05, 0.0
    )
    churn_prob += np.where(
        is_new & (recharge_frequency_30d >= 6),
        0.01, 0.0
    )

    # --------------------------------------------------
    # SIGNAL 6: Number portability — 1.4x multiplier
    # Ported in once = demonstrated willingness to switch
    # --------------------------------------------------
    churn_prob *= np.where(number_portability_flag == 1, 1.4, 1.0)

    # --------------------------------------------------
    # SIGNAL 7: Customer service contacts — escalation conditional
    #
    # contacts >= 5 (any status)
    #   → +0.08 (five contacts = something seriously wrong)
    # contacts >= 3 AND complaint = 1
    #   → +0.10 (multiple contacts AND formal escalation = unresolved)
    # contacts >= 3 AND complaint = 0
    #   → +0.03 (multiple contacts, no escalation — possibly resolved)
    #
    # Worst case: contacts >= 5 AND complaint = 1
    #   → +0.08 + +0.10 = +0.18 total
    #   (contacted 5+ times AND still filed a formal complaint)
    # --------------------------------------------------
    high_contacts = customer_service_contacts_30d >= 3
    very_high_contacts = customer_service_contacts_30d >= 5

    churn_prob += np.where(very_high_contacts, 0.08, 0.0)

    churn_prob += np.where(
        high_contacts & (network_complaint_flag == 1), 0.10, 0.0)
    churn_prob += np.where(
        high_contacts & (network_complaint_flag == 0), 0.03, 0.0)

    # --------------------------------------------------
    # SIGNAL 8: High value subscriber — protective effect
    # More invested in the relationship — higher switching cost
    # --------------------------------------------------
    churn_prob -= np.where(avg_recharge_value_naira > 1000, 0.05, 0.0)

    # --------------------------------------------------
    # SIGNAL 9T: USSD decay — traditional disengagement
    # A traditional subscriber not checking balance has mentally left
    # --------------------------------------------------
    churn_prob += np.where(
        is_traditional & (ussd_sessions_30d < 3), 0.11, 0.0
    )

    # --------------------------------------------------
    # SIGNAL 10T: Voice Only bundle + complaint
    # Traditional + Voice Only + complaint = network quality churn
    # Almost certainly experiencing coverage or call quality issues
    # --------------------------------------------------
    churn_prob += np.where(
        is_traditional &
        (bundle_type == 'Voice Only') &
        (network_complaint_flag == 1),
        0.05, 0.0
    )

    # --------------------------------------------------
    # SIGNAL 11T: Scratch card frequency decay — rural traditional
    #
    # Only applies to: traditional + scratch card + non-urban state
    # Logic: the CHANNEL is not the risk — declining FREQUENCY is
    #
    # Tier 1 — consistent buyer (frequency >= 4)
    #   → +0.005 (loyal rural subscriber — near-zero risk)
    # Tier 2 — slowing (frequency 2–3)
    #   → +0.02  (watch signal — behaviour changing)
    # Tier 3 — collapsed (frequency < 2)
    #   → +0.06  (stopped buying scratch cards — disengaged)
    # Tier 3 confirmed — collapsed AND days_since > 21
    #   → +0.04 additional (double confirmation — subscriber has left)
    # --------------------------------------------------
    is_scratch_rural = (
        is_traditional &
        (recharge_channel == 'Scratch Card') &
        ~is_urban_state
    )

    churn_prob += np.where(
        is_scratch_rural & (recharge_frequency_30d >= 4),
        0.005, 0.0
    )
    churn_prob += np.where(
        is_scratch_rural &
        (recharge_frequency_30d >= 2) &
        (recharge_frequency_30d < 4),
        0.02, 0.0
    )
    churn_prob += np.where(
        is_scratch_rural & (recharge_frequency_30d < 2),
        0.06, 0.0
    )
    # Tier 3 double confirmation
    churn_prob += np.where(
        is_scratch_rural &
        (recharge_frequency_30d < 2) &
        (days_since_last_recharge > 21),
        0.04, 0.0
    )

    # --------------------------------------------------
    # SIGNAL 12D: Social media bundle — tiered digital
    #
    # Most specific conditions checked first.
    # Only one tier applies per subscriber.
    #
    # Tier A: new + barely recharging
    #   tenure < 12 AND frequency < 3 → +0.10 (very high risk)
    # Tier B: competitor SIM present
    #   competitor_sim = 1            → +0.09 (one offer away)
    # Tier C: long tenure + consistent
    #   tenure >= 24 AND frequency >= 4 → +0.01 (preference not problem)
    # Tier D: all other cases
    #   baseline                       → +0.06
    #
    # Logic: these are mutually exclusive — we use elif ordering
    # via nested np.where to ensure only one tier fires per row
    # --------------------------------------------------
    is_social = is_digital & (bundle_type == 'Social Media Bundle')

    social_signal = np.where(
        is_social & (tenure_months < 12) & (recharge_frequency_30d < 3),
        0.10,
        np.where(
        is_social & (competitor_sim_detected == 1),
        0.09,
        np.where(
        is_social & (tenure_months >= 24) & (recharge_frequency_30d >= 4),
        0.01,
        np.where(
        is_social,
        0.06,
        0.0))))

    churn_prob += social_signal

    # --------------------------------------------------
    # SIGNAL 13D: Low app sessions — digital disengagement
    # A digital subscriber not using their primary self-service
    # channel is showing early disengagement
    # Note: this signal is intentionally weak standalone —
    # its full weight comes from Signal 14D stacking
    # --------------------------------------------------
    churn_prob += np.where(
        is_digital & (app_sessions_30d < 2), 0.05, 0.0
    )

    # --------------------------------------------------
    # SIGNAL 14D: Low data consumption — stacked digital
    #
    # Most specific condition checked first — mutually exclusive tiers
    #
    # Tier 1: data < 500 AND app < 2 AND frequency < 3
    #   → +0.12 (all three digital channels quiet simultaneously)
    # Tier 2: data < 500 AND app < 2
    #   → +0.07 (two digital disengagement signals)
    # Tier 3: data < 500 alone
    #   → +0.03 (single signal — could be temporary)
    # --------------------------------------------------
    low_data = is_digital & (data_mb_consumed_30d < 500)
    low_app  = app_sessions_30d < 2
    low_freq_digital = recharge_frequency_30d < 3

    data_signal = np.where(
        low_data & low_app & low_freq_digital,
        0.12,
        np.where(
        low_data & low_app,
        0.07,
        np.where(
        low_data,
        0.03,
        0.0)))

    churn_prob += data_signal

    # --------------------------------------------------------
    # SECTION 18 — SEASONAL ADJUSTMENTS
    # January:  post-holiday cash squeeze  → +8pp
    # July–Aug: school fees pressure       → +5pp
    # --------------------------------------------------------

    months = pd.DatetimeIndex(observation_months).month
    churn_prob += np.where(months == 1, 0.08, 0.0)
    churn_prob += np.where((months == 7) | (months == 8), 0.05, 0.0)

    # --------------------------------------------------------
    # SECTION 19 — STATE-LEVEL ADJUSTMENTS
    # High-competition urban states: +7pp
    # Urban digital subscribers: additional +3pp
    # (More competitor exposure + higher smartphone penetration)
    # --------------------------------------------------------

    churn_prob += np.where(is_urban_state, 0.07, 0.0)
    churn_prob += np.where(is_digital & is_urban_state, 0.03, 0.0)

    # --------------------------------------------------------
    # SECTION 20 — CLIP AND DRAW FINAL CHURN LABEL
    # --------------------------------------------------------

    # Clip to valid probability range [0.01, 0.95]
    # No subscriber is exactly 0% or 100% — always uncertainty
    churn_prob = np.clip(churn_prob, 0.01, 0.95)

    # Draw binary label from each subscriber's individual probability
    # np.random.binomial(1, p) = 1 with probability p
    # This is where a continuous probability becomes a hard 0 or 1
    churn_label = np.random.binomial(1, churn_prob)

    # --------------------------------------------------------
    # SECTION 21 — ASSEMBLE THE DATAFRAME
    # Column order follows locked schema exactly
    # --------------------------------------------------------

    df = pd.DataFrame({
        'subscriber_id'                : subscriber_ids,
        'observation_month'            : observation_months,
        'subscriber_segment'           : subscriber_segment,
        'days_since_last_recharge'     : days_since_last_recharge,
        'recharge_frequency_30d'       : recharge_frequency_30d,
        'avg_recharge_value_naira'     : avg_recharge_value_naira,
        'ussd_sessions_30d'            : ussd_sessions_30d,
        'app_sessions_30d'             : app_sessions_30d,
        'data_mb_consumed_30d'         : data_mb_consumed_30d,
        'voice_minutes_30d'            : voice_minutes_30d,
        'sms_count_30d'                : sms_count_30d,
        'network_complaint_flag'       : network_complaint_flag,
        'competitor_sim_detected'      : competitor_sim_detected,
        'number_portability_flag'      : number_portability_flag,
        'tenure_months'                : tenure_months,
        'recharge_channel'             : recharge_channel,
        'bundle_type'                  : bundle_type,
        'customer_service_contacts_30d': customer_service_contacts_30d,
        'tariff_plan'                  : tariff_plans,
        'state'                        : states,
        'churn_label'                  : churn_label
    })

    print(f"Dataset generated successfully.")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

    return df


# --- Run the function ---
df = generate_mtn_dataset(N_SUBSCRIBERS, random_seed=RANDOM_SEED)

# --- Quick preview ---
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())

Generating 500,000 subscriber records...
This may take 30–60 seconds. Please wait.
Dataset generated successfully.
Shape: 500,000 rows × 21 columns

First 3 rows:
  subscriber_id observation_month subscriber_segment  days_since_last_recharge  recharge_frequency_30d  avg_recharge_value_naira  ussd_sessions_30d  app_sessions_30d  data_mb_consumed_30d  voice_minutes_30d  sms_count_30d  network_complaint_flag  competitor_sim_detected  number_portability_flag  tenure_months recharge_channel          bundle_type  customer_service_contacts_30d  tariff_plan   state  churn_label
0  MTN-75682867        2024-06-01            Digital                        25                       4                   1376.28                  8                11               8121.67             103.72              6                       0                        0                        0             36    Bank Transfer  Social Media Bundle                              2  XtraSpecial  Jigawa            0
1  MTN-66

In [6]:
# ============================================================
# CELL 4 — DATA QUALITY CHECKS
# Validates the generated dataset before any analysis begins.
# Every check uses assert — if anything fails, the notebook
# stops immediately with a clear error message.
# This is standard professional practice — never skip this.
# ============================================================

print("=" * 60)
print("DATA QUALITY REPORT — NGA_MTN_Subscribers_500K")
print("=" * 60)

# --------------------------------------------------------
# CHECK 1 — SHAPE
# Must be exactly 500,000 rows and 21 columns
# --------------------------------------------------------

print("\n[CHECK 1] Shape")
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")

assert df.shape[0] == 500_000, \
    f"FAIL: Expected 500,000 rows, got {df.shape[0]:,}"
assert df.shape[1] == 21, \
    f"FAIL: Expected 21 columns, got {df.shape[1]}"

print("  STATUS  : PASSED ✓")

# --------------------------------------------------------
# CHECK 2 — DATA TYPES
# Pandas 3.0 stores string columns as dtype='str'
# Older Pandas stored them as dtype=object
# We accept both to stay version-compatible
# --------------------------------------------------------

print("\n[CHECK 2] Data Types")
print(df.dtypes.to_string())

# Helper: acceptable string dtypes across Pandas versions
STRING_DTYPES = [object, 'str', 'string']

# String columns
assert df['subscriber_id'].dtype in STRING_DTYPES, \
    f"FAIL: subscriber_id should be string, got {df['subscriber_id'].dtype}"

assert df['subscriber_segment'].dtype in STRING_DTYPES, \
    f"FAIL: subscriber_segment should be string, got {df['subscriber_segment'].dtype}"

assert df['recharge_channel'].dtype in STRING_DTYPES, \
    f"FAIL: recharge_channel should be string, got {df['recharge_channel'].dtype}"

assert df['bundle_type'].dtype in STRING_DTYPES, \
    f"FAIL: bundle_type should be string, got {df['bundle_type'].dtype}"

assert df['tariff_plan'].dtype in STRING_DTYPES, \
    f"FAIL: tariff_plan should be string, got {df['tariff_plan'].dtype}"

assert df['state'].dtype in STRING_DTYPES, \
    f"FAIL: state should be string, got {df['state'].dtype}"

# Datetime column
assert str(df['observation_month'].dtype).startswith('datetime'), \
    f"FAIL: observation_month should be datetime, got {df['observation_month'].dtype}"

# Integer columns
INT_DTYPES = ['int32', 'int64']

assert df['days_since_last_recharge'].dtype in INT_DTYPES, \
    f"FAIL: days_since_last_recharge should be int, got {df['days_since_last_recharge'].dtype}"

assert df['recharge_frequency_30d'].dtype in INT_DTYPES, \
    f"FAIL: recharge_frequency_30d should be int, got {df['recharge_frequency_30d'].dtype}"

assert df['ussd_sessions_30d'].dtype in INT_DTYPES, \
    f"FAIL: ussd_sessions_30d should be int, got {df['ussd_sessions_30d'].dtype}"

assert df['app_sessions_30d'].dtype in INT_DTYPES, \
    f"FAIL: app_sessions_30d should be int, got {df['app_sessions_30d'].dtype}"

assert df['sms_count_30d'].dtype in INT_DTYPES, \
    f"FAIL: sms_count_30d should be int, got {df['sms_count_30d'].dtype}"

assert df['network_complaint_flag'].dtype in INT_DTYPES, \
    f"FAIL: network_complaint_flag should be int, got {df['network_complaint_flag'].dtype}"

assert df['competitor_sim_detected'].dtype in INT_DTYPES, \
    f"FAIL: competitor_sim_detected should be int, got {df['competitor_sim_detected'].dtype}"

assert df['number_portability_flag'].dtype in INT_DTYPES, \
    f"FAIL: number_portability_flag should be int, got {df['number_portability_flag'].dtype}"

assert df['tenure_months'].dtype in INT_DTYPES, \
    f"FAIL: tenure_months should be int, got {df['tenure_months'].dtype}"

assert df['customer_service_contacts_30d'].dtype in INT_DTYPES, \
    f"FAIL: customer_service_contacts_30d should be int, got {df['customer_service_contacts_30d'].dtype}"

assert df['churn_label'].dtype in INT_DTYPES, \
    f"FAIL: churn_label should be int, got {df['churn_label'].dtype}"

# Float columns
FLOAT_DTYPES = ['float32', 'float64']

assert df['avg_recharge_value_naira'].dtype in FLOAT_DTYPES, \
    f"FAIL: avg_recharge_value_naira should be float, got {df['avg_recharge_value_naira'].dtype}"

assert df['data_mb_consumed_30d'].dtype in FLOAT_DTYPES, \
    f"FAIL: data_mb_consumed_30d should be float, got {df['data_mb_consumed_30d'].dtype}"

assert df['voice_minutes_30d'].dtype in FLOAT_DTYPES, \
    f"FAIL: voice_minutes_30d should be float, got {df['voice_minutes_30d'].dtype}"

print("\n  STATUS  : PASSED ✓")

# --------------------------------------------------------
# CHECK 3 — NULL VALUES
# We generated this data — there should be zero nulls.
# Any null means something went wrong in Cell 3.
# --------------------------------------------------------

print("\n[CHECK 3] Null Values")
null_counts  = df.isnull().sum()
total_nulls  = null_counts.sum()

print(f"  Total null values across all columns: {total_nulls}")

if total_nulls > 0:
    print("\n  Columns with nulls:")
    print(null_counts[null_counts > 0].to_string())

assert total_nulls == 0, \
    f"FAIL: Found {total_nulls} null values — check Cell 3 generation logic"

print("  STATUS  : PASSED ✓")

# --------------------------------------------------------
# CHECK 4 — VALUE RANGES
# Confirm every numeric column is within its defined range.
# We print min and max for every column so we can spot
# anything suspicious even if it passes the assert.
# --------------------------------------------------------

print("\n[CHECK 4] Value Ranges")
print(f"  {'Column':<35} {'Min':>10} {'Max':>12} {'Status'}")
print(f"  {'-'*35} {'-'*10} {'-'*12} {'-'*8}")

# (column_name, min_allowed, max_allowed)
range_checks = [
    ('days_since_last_recharge',       0,     90),
    ('recharge_frequency_30d',         0,     30),
    ('avg_recharge_value_naira',       50,  5000),
    ('ussd_sessions_30d',              0,    200),
    ('app_sessions_30d',               0,     50),
    ('data_mb_consumed_30d',           0,  50000),
    ('voice_minutes_30d',              0,   1000),
    ('sms_count_30d',                  0,    300),
    ('network_complaint_flag',         0,      1),
    ('competitor_sim_detected',        0,      1),
    ('number_portability_flag',        0,      1),
    ('tenure_months',                  1,    120),
    ('customer_service_contacts_30d',  0,     10),
    ('churn_label',                    0,      1),
]

all_ranges_passed = True
for col, min_val, max_val in range_checks:
    actual_min = df[col].min()
    actual_max = df[col].max()
    passed     = (actual_min >= min_val) and (actual_max <= max_val)
    status     = "PASSED ✓" if passed else "FAILED ✗"
    print(f"  {col:<35} {actual_min:>10.2f} {actual_max:>12.2f} {status}")
    if not passed:
        all_ranges_passed = False
        print(f"    Expected range: [{min_val}, {max_val}]")

assert all_ranges_passed, \
    "FAIL: One or more columns have values outside their expected range"

print(f"\n  STATUS  : PASSED ✓")

# --------------------------------------------------------
# CHECK 5 — CATEGORICAL VALUE VALIDATION
# Confirm only valid categories exist in each column
# --------------------------------------------------------

print("\n[CHECK 5] Categorical Values")

# subscriber_segment
valid_segments  = {'Traditional', 'Digital'}
actual_segments = set(df['subscriber_segment'].unique())
assert actual_segments == valid_segments, \
    f"FAIL: subscriber_segment unexpected values: {actual_segments - valid_segments}"
print(f"  subscriber_segment    : {sorted(actual_segments)} ✓")

# tariff_plan
valid_plans  = set(TARIFF_PLANS)
actual_plans = set(df['tariff_plan'].unique())
assert actual_plans == valid_plans, \
    f"FAIL: tariff_plan unexpected values: {actual_plans - valid_plans}"
print(f"  tariff_plan           : {sorted(actual_plans)} ✓")

# recharge_channel
valid_channels  = {'Scratch Card', 'Agent', 'USSD', 'Bank Transfer', 'MTN App'}
actual_channels = set(df['recharge_channel'].unique())
assert actual_channels == valid_channels, \
    f"FAIL: recharge_channel unexpected values: {actual_channels - valid_channels}"
print(f"  recharge_channel      : {sorted(actual_channels)} ✓")

# bundle_type
valid_bundles  = {'Voice Only', 'Data Only', 'Voice+Data Combo', 'Social Media Bundle'}
actual_bundles = set(df['bundle_type'].unique())
assert actual_bundles == valid_bundles, \
    f"FAIL: bundle_type unexpected values: {actual_bundles - valid_bundles}"
print(f"  bundle_type           : {sorted(actual_bundles)} ✓")

# state
valid_states  = set(NIGERIAN_STATES)
actual_states = set(df['state'].unique())
assert actual_states == valid_states, \
    f"FAIL: state unexpected values: {actual_states - valid_states}"
print(f"  state                 : all 37 states present ✓")

print(f"\n  STATUS  : PASSED ✓")

# --------------------------------------------------------
# CHECK 6 — CHURN RATE AND SEGMENT DISTRIBUTION
# Churn rate will be higher than the 5% base rate because
# the signal boosting in Cell 3 pushes many subscribers
# into higher probability buckets before the binary draw.
# Expected range: 10%–30%
# --------------------------------------------------------

print("\n[CHECK 6] Churn Rate and Segment Distribution")

overall_churn_rate = df['churn_label'].mean() * 100
churned_count      = df['churn_label'].sum()
retained_count     = len(df) - churned_count

print(f"\n  Churn label distribution:")
print(f"  Churned  (1) : {churned_count:>8,}  ({overall_churn_rate:.2f}%)")
print(f"  Retained (0) : {retained_count:>8,}  ({100 - overall_churn_rate:.2f}%)")

# Segment distribution
segment_counts = df['subscriber_segment'].value_counts()
trad_pct       = segment_counts.get('Traditional', 0) / len(df) * 100
dig_pct        = segment_counts.get('Digital',     0) / len(df) * 100

print(f"\n  Segment distribution:")
print(f"  Traditional  : {segment_counts.get('Traditional', 0):>8,}  ({trad_pct:.1f}%)")
print(f"  Digital      : {segment_counts.get('Digital',     0):>8,}  ({dig_pct:.1f}%)")

# Churn rate by segment
trad_churn = df[df['subscriber_segment'] == 'Traditional']['churn_label'].mean() * 100
dig_churn  = df[df['subscriber_segment'] == 'Digital'    ]['churn_label'].mean() * 100

print(f"\n  Churn rate by segment:")
print(f"  Traditional churn rate : {trad_churn:.2f}%")
print(f"  Digital churn rate     : {dig_churn:.2f}%")

# Churn rate by high-churn states vs rest
high_churn_mask    = df['state'].isin(HIGH_CHURN_STATES)
urban_churn_rate   = df[high_churn_mask ]['churn_label'].mean() * 100
rural_churn_rate   = df[~high_churn_mask]['churn_label'].mean() * 100

print(f"\n  Churn rate by geography:")
print(f"  Urban high-churn states : {urban_churn_rate:.2f}%")
print(f"  All other states        : {rural_churn_rate:.2f}%")

# Assert churn rate is in expected range
assert 10 <= overall_churn_rate <= 30, \
    f"FAIL: Overall churn rate {overall_churn_rate:.2f}% outside " \
    f"expected range [10%, 30%]. Check signal logic in Cell 3."

# Assert segment split is within 3pp of target
assert abs(trad_pct - 45) <= 3, \
    f"FAIL: Traditional segment {trad_pct:.1f}%, expected ~45%"
assert abs(dig_pct - 55) <= 3, \
    f"FAIL: Digital segment {dig_pct:.1f}%, expected ~55%"

# Assert digital churn is higher than traditional
# (Urban digital subscribers have more competitor exposure)
assert dig_churn > trad_churn, \
    f"FAIL: Digital churn ({dig_churn:.2f}%) should exceed " \
    f"Traditional churn ({trad_churn:.2f}%) — check segment signals"

print(f"\n  STATUS  : PASSED ✓")

# --------------------------------------------------------
# CHECK 7 — SUBSCRIBER ID FORMAT
# All IDs must match MTN-XXXXXXXX (MTN- + 8 digits)
# --------------------------------------------------------

print("\n[CHECK 7] Subscriber ID Format")

id_format_check = df['subscriber_id'].str.match(r'^MTN-\d{8}$')
malformed_count = (~id_format_check).sum()

print(f"  Total IDs             : {len(df):,}")
print(f"  Correctly formatted   : {id_format_check.sum():,}")
print(f"  Malformed IDs         : {malformed_count}")

assert malformed_count == 0, \
    f"FAIL: {malformed_count} subscriber IDs do not match MTN-XXXXXXXX format"

print(f"  STATUS  : PASSED ✓")

# --------------------------------------------------------
# FINAL SUMMARY
# --------------------------------------------------------

print("\n" + "=" * 60)
print("DATA QUALITY REPORT — SUMMARY")
print("=" * 60)
print(f"  Check 1 — Shape              : PASSED ✓")
print(f"  Check 2 — Data Types         : PASSED ✓")
print(f"  Check 3 — Null Values        : PASSED ✓")
print(f"  Check 4 — Value Ranges       : PASSED ✓")
print(f"  Check 5 — Categorical Values : PASSED ✓")
print(f"  Check 6 — Churn Distribution : PASSED ✓")
print(f"  Check 7 — ID Format          : PASSED ✓")
print("=" * 60)
print(f"  Dataset is clean and ready for EDA.")
print("=" * 60)

DATA QUALITY REPORT — NGA_MTN_Subscribers_500K

[CHECK 1] Shape
  Rows    : 500,000
  Columns : 21
  STATUS  : PASSED ✓

[CHECK 2] Data Types
subscriber_id                               str
observation_month                datetime64[us]
subscriber_segment                          str
days_since_last_recharge                  int64
recharge_frequency_30d                    int32
avg_recharge_value_naira                float64
ussd_sessions_30d                         int32
app_sessions_30d                          int32
data_mb_consumed_30d                    float64
voice_minutes_30d                       float64
sms_count_30d                             int32
network_complaint_flag                    int32
competitor_sim_detected                   int32
number_portability_flag                   int32
tenure_months                             int64
recharge_channel                            str
bundle_type                                 str
customer_service_contacts_30d             